# Day 10 · Exercise 3: Build a PromptTemplate

**What you'll build:** `build_prompt_template(name: str, template_str: str, description: str) -> dict` — a factory that constructs a self-documenting PromptTemplate dataclass (with `name`, `template_str`, `required_vars`, and `description` fields) and returns it as a plain dict for easy inspection.

**Why it matters:** Wrapping a bare `string.Template` in a typed dataclass with an explicit `required_vars` contract means you catch missing variables at configuration time — not mid-request — and every template becomes readable, loggable, and testable without touching the model.

## Your Implementation

In [ ]:
from dataclasses import dataclass, asdict  # asdict(instance) converts a dataclass to a plain dict
from string import Template
import re


@dataclass
class PromptTemplate:
    """A self-documenting, validatable prompt template.

    Attributes:
        name: Unique identifier used in error messages and logging.
        template_str: Raw template text with $variable placeholders.
        required_vars: Variables the caller must supply before render.
        description: One sentence explaining what the filled template produces.
    """
    name: str
    template_str: str
    required_vars: list
    description: str

    def validate(self, values: dict) -> None:
        """Raise ValueError listing every required variable not in values."""
        missing = [v for v in self.required_vars if v not in values]
        if missing:
            raise ValueError(
                f"PromptTemplate '{self.name}' is missing required "
                f"variable(s): {missing}"
            )

    def render(self, values: dict) -> str:
        """Validate values, then return the fully substituted prompt string."""
        self.validate(values)
        return Template(self.template_str).substitute(values)


def build_prompt_template(name: str, template_str: str, description: str) -> dict:
    """Build a PromptTemplate from a name, template string, and description.

    Extracts required_vars automatically from the dollar-sign placeholders
    found in template_str, then constructs a PromptTemplate dataclass and
    returns it as a plain dict (via dataclasses.asdict) for easy inspection.
    asdict(instance) converts every field of the dataclass into a key-value
    pair in a regular Python dict — no model call, no serialisation library.

    Args:
        name: Unique identifier for the template (used in error messages).
        template_str: Template text with $variable placeholders
            (e.g. "Summarise $text in $max_sentences sentences.").
        description: One sentence explaining what the filled template produces.

    Returns:
        A dict with keys: 'name', 'template_str', 'required_vars', 'description'.
        'required_vars' is a sorted list of unique placeholder names extracted
        from template_str.

    Example:
        result = build_prompt_template(
            name="summarise_brief",
            template_str="Summarise $text in $max_sentences sentences.",
            description="Summarise a passage in a fixed sentence count.",
        )
        # result['required_vars'] == ['max_sentences', 'text']
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and returns a dict
    try:
        result = build_prompt_template(
            name="test_tmpl",
            template_str="Hello $name, you have $count messages.",
            description="Greet a user and show their message count.",
        )
        assert isinstance(result, dict), f"expected dict, got {type(result).__name__}"
        print(f"{_PASS} Check 1/{total}: build_prompt_template returns a dict")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1/{total}: {e}")
        return

    # Check 2: dict contains all four required keys
    try:
        required_keys = {"name", "template_str", "required_vars", "description"}
        assert required_keys.issubset(result.keys()), (
            f"missing keys: {required_keys - set(result.keys())}"
        )
        print(f"{_PASS} Check 2/{total}: result dict has all four keys (name, template_str, required_vars, description)")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 2/{total}: {e}")

    # Check 3: required_vars are extracted correctly from template_str
    try:
        assert sorted(result["required_vars"]) == ["count", "name"], (
            f"expected ['count', 'name'], got {sorted(result['required_vars'])}"
        )
        print(f"{_PASS} Check 3/{total}: required_vars correctly extracted as ['count', 'name']")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 3/{total}: {e}")

    # Check 4: validate() raises ValueError listing the missing variable
    try:
        pt = PromptTemplate(**result) if not isinstance(result.get("required_vars"), list) else PromptTemplate(
            name=result["name"],
            template_str=result["template_str"],
            required_vars=result["required_vars"],
            description=result["description"],
        )
        try:
            pt.validate({"name": "Alice"})   # 'count' is missing
            print(f"{_FAIL} Check 4/{total}: validate() should have raised ValueError")
        except ValueError as ve:
            assert "count" in str(ve), f"error message did not name 'count': {ve}"
            print(f"{_PASS} Check 4/{total}: validate() raises ValueError naming the missing variable")
            score += 1
    except Exception as e:
        print(f"{_FAIL} Check 4/{total}: {e}")

    # Check 5: render() returns a correctly substituted string
    try:
        rendered = pt.render({"name": "Alice", "count": "3"})
        assert "Alice" in rendered, f"'Alice' not found in rendered output: {rendered!r}"
        assert "3" in rendered, f"'3' not found in rendered output: {rendered!r}"
        print(f"{_PASS} Check 5/{total}: render() returns the fully substituted prompt string")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 5/{total}: {e}")

    print()
    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed." + (" Keep going!" if score < total else ""))


_run_checks()

## Bonus Challenge

Foreshadowing Day 11 (prompt libraries): extend `build_prompt_template` so it also stores the returned dict in a module-level registry `dict[str, dict]` keyed by `name`. Then write a `get_template(name: str) -> PromptTemplate` helper that looks up the registry and reconstructs the dataclass. This is the seed of a full prompt-library module — the pattern you will build in Lesson 4.

```python
TEMPLATE_REGISTRY: dict = {}

def get_template(name: str) -> PromptTemplate:
    entry = TEMPLATE_REGISTRY[name]   # KeyError if not registered
    return PromptTemplate(**entry)
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
from dataclasses import dataclass, asdict
from string import Template
import re


@dataclass
class PromptTemplate:
    name: str
    template_str: str
    required_vars: list
    description: str

    def validate(self, values: dict) -> None:
        missing = [v for v in self.required_vars if v not in values]
        if missing:
            raise ValueError(
                f"PromptTemplate '{self.name}' is missing required "
                f"variable(s): {missing}"
            )

    def render(self, values: dict) -> str:
        self.validate(values)
        return Template(self.template_str).substitute(values)


def build_prompt_template(name: str, template_str: str, description: str) -> dict:
    # Extract every $identifier placeholder from the template string.
    # The pattern matches $word or ${word}; we deduplicate and sort for
    # a stable, predictable required_vars list.
    placeholders = re.findall(r"\$\{?(\w+)\}?", template_str)
    required_vars = sorted(set(placeholders))

    pt = PromptTemplate(
        name=name,
        template_str=template_str,
        required_vars=required_vars,
        description=description,
    )
    return asdict(pt)
```

**Why this works:** `re.findall(r"\$\{?(\w+)\}?", template_str)` captures every dollar-sign identifier in both `$var` and `${var}` forms — the same two syntaxes `string.Template` itself recognises. Deduplicating with `set` and sorting with `sorted` produces a stable list that the automated checks can compare without order mattering. `dataclasses.asdict` converts the dataclass to a plain dict in one call, which is easier to inspect, serialise, or store than the dataclass object itself.
</details>